In this notebook will be using the same class methods created erlier and change the code as required for the streaming read and write


In [0]:
class word_counts():
    def __init__(self):
        self.base_directory='/Volumes/workspace/word_count/sample/text/'
        # path should not be file level and it should be folder level

#__init__ is also called as the constructor of the class which should be created at each class
        
    def read_file(self):
        lines =spark.readStream.text(self.base_directory)
        print('displaying the read data')
        #  In the above line the read is replaced with readSteam for the streaming requirement
        return lines
    def quality_words(self,lines):
        from pyspark.sql.functions import split,explode,split,lower,trim
        raw_words=lines.select(explode(split(lines.value," ")).alias('words'))
        good_words=raw_words.select(lower(trim(raw_words.words)).alias('good_words'))
        return good_words
    def word_counts(self,good_words):
        word_counts=good_words.groupBy("good_words").count()
        return word_counts
    def final_write(self,word_counts):
        return(word_counts.writeStream
        .format('delta')
        .option ('checkpointLocation','/Volumes/workspace/word_count/sample/checkpoint/wordcount')
        .outputMode('Complete')
        .trigger(once=True)
        #.trigger(processingTime='2 minutes')  # triggers every 2 minutes
        .toTable('word_counts_strem'))

        # the above write fucntion has been update with write to writeSteam, and mode to outputMode and saveAsTable to toTable. It also requires the checkpoint directory to be mentioned only in the write part. 
        # The entire writestream has to be return value and it has to be put into return statement
    def run_load(self):
        print("Executing the code ",end=" ")
        lines=self.read_file()
        good_words=self.quality_words(lines)
        word_counts=self.word_counts(good_words)
        sQuery=self.final_write(word_counts)
        print("Done")
        return(sQuery)
    def assertcount(self,expected_count):
        sQuery=self.run_load()
        import time
        SleepTime=30
        print('Its loading waiting for 30 sec')
        print('Starting the test --- ', end=" ")
        time.sleep(SleepTime)
        actual_count=spark.sql("select count from word_counts_strem where good_words='on'").collect()[0][0]
        print("actual_count",actual_count)
        assert actual_count==expected_count ,f"Test failed!! the count is {actual_count} but the provided count is {expected_count}"
        print('Test completed')
        sQuery.stop()

        
        

In [0]:
word_count_obj=word_counts()
word_count_obj.assertcount(expected_count=4)